In [ ]:
import re
import time
import requests
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*"
}

def extract_ids_from_url(url):
    """Tự động lấy product_id và spid từ URL Tiki"""
    product_match = re.search(r"-p(\d+)", url)
    spid_match = re.search(r"spid=(\d+)", url)
    product_id = product_match.group(1) if product_match else None
    spid = spid_match.group(1) if spid_match else None
    return product_id, spid


def check_valid_id(pid, use_spid=False):
    """Kiểm tra ID có review thật hay không"""
    url_type = "spid" if use_spid else "product_id"
    url = f"https://tiki.vn/api/v2/reviews?{url_type}={pid}&limit=1&page=1"
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        return False
    try:
        data = res.json()
    except ValueError:
        return False
    return len(data.get("data", [])) > 0


def crawl_reviews(pid, use_spid=False, max_pages=10):
    """Crawl tất cả review từ ID hợp lệ"""
    url_type = "spid" if use_spid else "product_id"
    all_reviews = []

    for page in range(1, max_pages + 1):
        url = f"https://tiki.vn/api/v2/reviews?{url_type}={pid}&limit=20&page={page}&include=comments,photos"
        res = requests.get(url, headers=headers)
        try:
            data = res.json()
        except ValueError:
            break
        reviews = data.get("data", [])
        if not reviews:
            break
        for item in reviews:
            all_reviews.append({
                "id": item.get("id"),
                "rating": item.get("rating"),
                "title": item.get("title"),
                "content": item.get("content"),
                "name": (item.get("created_by") or {}).get("name"), # FIX: Handle created_by being None
                "created_at": item.get("created_at"),
                "thank_count": item.get("thank_count"),
                "has_photo": len(item.get("images", [])) > 0
            })
        print(f"Đã crawl trang {page} ({len(reviews)} bình luận)")
        time.sleep(2) # Increased sleep time for pages

    return pd.DataFrame(all_reviews)


def auto_crawl_from_url(url):
    """Quy trình tự động từ URL đến crawl"""
    product_id, spid = extract_ids_from_url(url)
    print(f"🔍 product_id = {product_id}, spid = {spid}")

    # Thử product_id trước
    valid_type = None
    if product_id and check_valid_id(product_id, use_spid=False):
        valid_type = ("product_id", product_id)
    elif spid and check_valid_id(spid, use_spid=True):
        valid_type = ("spid", spid)
    else:
        print("❌ Không tìm thấy review cho cả product_id và spid.")
        return None

    print(f"✅ Dùng {valid_type[0]} = {valid_type[1]} để crawl reviews...")
    df = crawl_reviews(valid_type[1], use_spid=(valid_type[0] == "spid"))
    df.to_csv("tiki_reviews.csv", index=False, encoding="utf-8-sig")
    print(f"📁 Đã lưu tiki_reviews.csv với {len(df)} bình luận.")
    return df

# 👉 Ví dụ chạy thử:
urls = [
    "https://tiki.vn/apple-iphone-14-p197214029.html?spid=197214037",
    "https://tiki.vn/lo-nuong-chan-khong-dien-tu-lock-lock-ejf291blk-10l-hang-chinh-hang-p87960372.html?spid=87960373",
    "https://tiki.vn/sach-deep-work-lam-ra-lam-choi-ra-choi-tai-ban-p101782486.html?spid=101782488",
    "https://tiki.vn/sach-tri-thong-minh-tren-giuong-p93734071.html?spid=93734072",
    "https://tiki.vn/sach-noi-chuyen-la-ban-nang-giu-mieng-la-tu-duong-im-lang-la-tri-tue-p277728224.html?spid=212356653",
    "https://tiki.vn/sach-dan-dat-mot-bay-soi-hay-chan-mot-dan-cuu-p275406600.html?spid=275406603",
    "http://tiki.vn/sach-khi-moi-dieu-khong-nhu-y-p276960030.html?spid=276960033",
    "https://tiki.vn/sach-tam-ly-hoc-ve-tien-p273819808.html?spid=75953558",
    "http://tiki.vn/sach-dan-dat-mot-bay-soi-hay-chan-mot-dan-cuu-p275406600.html?spid=275406603",
    "https://tiki.vn/sach-osho-cam-xuc-p184070223.html?spid=184070224",
    "https://tiki.vn/sach-tu-duy-nhanh-va-cham-tai-ban-p276388548.html?spid=214186190",
    "https://tiki.vn/sach-con-duong-chang-may-ai-di-p274363543.html?spid=274363544",
    "https://tiki.vn/series-cac-tap-attack-on-titan-p274031576.html?spid=274607837",
    "https://tiki.vn/dien-thoai-samsung-galaxy-a26-5g-8-128gb-mat-lung-kinh-ai-circle-to-search-camera-hdr-chup-dem-sang-ro-hang-chinh-hang-p277777809.html?spid=277777811",
    "https://tiki.vn/may-tinh-bang-ipad-gen-11-a16-wifi-p277619147.html?spid=277619155",
    "https://tiki.vn/dien-thoai-xiaomi-redmi-note-14-6gb-128gb-hang-chinh-hang-p277007015.html?spid=277007021",
    "https://tiki.vn/dien-thoai-samsung-galaxy-a36-5g-hang-chinh-hang-p277466537.html?spid=277466541",
    "https://tiki.vn/apple-iphone-16-pro-max-p276109904.html?spid=276109953",
    "https://tiki.vn/dien-thoai-oppo-a58-6gb-128gb-hang-chinh-hang-p270975124.html?spid=270975155",
    "https://tiki.vn/may-tinh-bang-samsung-galaxy-tab-a9-wifi-4gb-64gb-hang-chinh-hang-p274101255.html?spid=274101974",
    "https://tiki.vn/may-doc-sach-all-new-kindle-paperwhite-5-11th-gen-hang-nhap-khau-p125182567.html?spid=125182569",
    "https://tiki.vn/dien-thoai-ban-kxts500-hang-chinh-hang-p163914808.html?spid=163914816",
    "https://tiki.vn/apple-iphone-13-hang-chinh-hang-p184059211.html?spid=123547403",
    "https://tiki.vn/dien-thoai-oppo-a3-6gb-128gb-hang-chinh-hang-p278219968.html?spid=278240505",
    "https://tiki.vn/may-doc-sach-kobo-libra-libra-colour-libra-2-hang-nhap-khau-p139632023.html?spid=275176274",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-gold-4-1400g-cho-tre-tu-2-6-tuoi-p186118355.html?spid=186118357",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-4-hop-thiec-1500g-danh-cho-tre-2-6-tuoi-p10710902.html?spid=23099882",
    "https://tiki.vn/sure-prevent-gold-ht-p273180229.html?spid=71080137",
    "https://tiki.vn/sua-bot-pha-san-growplus-suy-dinh-duong-do-thung-24-hop-180ml-p277597569.html?spid=277597575",
    "https://tiki.vn/ta-quan-huggies-skin-care-mega-jumbo-l100-4-mieng-voi-tram-tra-diu-da-p276109346.html?spid=276109347",
    "https://tiki.vn/ta-bim-quan-huggies-skin-care-mega-jumbo-m102-4-mieng-voi-tram-tra-diu-da-p276568601.html?spid=276568602",
    "https://tiki.vn/sua-bot-nestle-nan-supremepro-2-800g-nhap-khau-duc-voi-5hmo-dam-gentle-optipro-danh-cho-tre-tu-12-24-thang-tuoi-p275597414.html?spid=275597416",
    "https://tiki.vn/sua-bot-nestle-nan-supremepro-1-800g-nhap-khau-duc-voi-5hmo-dam-gentle-optipro-danh-cho-tre-tu-0-12-thang-tuoi-p275597411.html?spid=275597412",
    "https://tiki.vn/sua-meiji-lon-so-9-800g-noi-dia-nhat-cho-tre-tu-1-3-tuoi-p275358855.html?spid=275358856",
    "https://tiki.vn/combo-2-tam-lot-sunmate-cai-tien-moi-45-x-70cm-p199903945.html?spid=199903946",
    "https://tiki.vn/combo-2-ta-quan-nguoi-lon-sunmate-mem-mai-m9-9-mieng-p23957447.html?spid=23957448",
    "https://tiki.vn/combo-2-goi-ta-quan-nguoi-lon-sunmate-kho-thoang-l7-7-mieng-goi-p25027164.html?spid=25027165",
    "https://tiki.vn/sua-bot-nestle-nan-optipro-plus-3-800g-lon-voi-5hmo-ho-tro-tieu-hoa-de-khang-tri-nao-chieu-cao-be-1-2-tuoi-p274391512.html?spid=274391513",
    "https://tiki.vn/thung-3-goi-ta-bim-dan-so-sinh-huggies-skin-perfect-s-54-6-mieng-voi-2-vung-tham-giam-kich-ung-da-p274091852.html?spid=274091860",
    "https://tiki.vn/sua-bot-nan-infinipro-a2-thuy-si-1-400g-voi-su-ket-hop-doc-dao-giua-dam-quy-a2-va-5hmo-0-12-thang-p273923928.html?spid=273923929",
    "https://tiki.vn/sua-bot-nestle-nan-optipro-plus-1-800g-lon-voi-5hmo-san-xuat-tai-thuy-si-0-6-thang-p247427699.html?spid=247427700",
    "https://tiki.vn/sure-prevent-gold-ht-p273180229.html?spid=71080137",
    "https://tiki.vn/sua-bot-nestle-nan-optipro-plus-2-800g-lon-voi-5hmo-san-xuat-tai-thuy-si-6-12-thang-p247427686.html?spid=247427689",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-gold-4-850g-cho-tre-tu-2-6-tuoi-p186118363.html?spid=186118364",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-gold-1-800g-cho-tre-tu-0-6-thang-tuoi-p186118360.html?spid=186118362",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-gold-3-850g-cho-tre-tu-1-2-tuoi-p186118359.html?spid=186118361",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-gold-4-1400g-cho-tre-tu-2-6-tuoi-p186118355.html?spid=186118357",
    "https://tiki.vn/sua-bot-nestle-nan-infinipro-a2-1-cho-tre-tu-0-12-thang-tuoi-hop-800g-nhap-khau-nguyen-lon-tu-thuy-sy-p168663291.html?spid=168663292",
    "https://tiki.vn/sua-bot-nan-infinipro-a2-thuy-si-2-800g-voi-su-ket-hop-doc-dao-giua-dam-quy-a2-va-5hmo-12-24-thang-p168662079.html?spid=168662080",
    "https://tiki.vn/sua-bot-frisolac-gold-3-850g-danh-cho-tre-tu-1-2-tuoi-p88949150.html?spid=88949151",
    "https://tiki.vn/sua-bot-frisolac-gold-1-850g-danh-cho-tre-tu-0-6-thang-tuoi-p88924594.html?spid=88924595",
    "https://tiki.vn/sua-bot-frisolac-gold-2-850g-danh-cho-tre-tu-6-12-thang-tuoi-p88924596.html?spid=88924597",
    "https://tiki.vn/sua-bot-frisolac-gold-1-850g-danh-cho-tre-tu-0-6-thang-tuoi-p88924594.html?spid=88924595",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-3-hop-thiec-900g-p11341360.html?spid=13720953",
    "https://tiki.vn/sua-bot-vinamilk-dielac-alpha-step-3-hop-thiec-1500g-p10710898.html?spid=23100087",
    "https://tiki.vn/tai-nghe-baseus-encok-3-5mm-lateral-in-ear-wired-earphone-h17-hang-chinh-hang-p181185263.html?spid=271974472",
    "https://tiki.vn/kinh-cuong-luc-kingkong-co-khung-tu-dan-ngan-bui-loa-danh-cho-iphone-full-hop-dan-full-man-hang-chinh-hang-p227746564.html?spid=271829333",
    "https://tiki.vn/bo-sac-nhanh-cho-iphone-pd20w-hoco-c80-c80a-plus-2-co-ng-usb-va-type-c-ke-m-day-sa-c-type-c-sang-iphone-1m-sac-0-50-pin-trong-30-phut-ha-ng-chi-nh-ha-ng-p276827665.html?spid=205291720",
    "https://tiki.vn/headphone-tai-nghe-chup-tai-bluetooth-rockspace-o2-ket-noi-khong-day-co-mic-nghe-nhac-lien-tuc-15h-hang-chinh-hang-p135284107.html?spid=135284112",
    "https://tiki.vn/cap-sac-baseus-cafule-sac-nhanh-dung-cho-iphone-hang-chinh-hang-p142439870.html?spid=205291518",
    "https://tiki.vn/chuot-co-day-logitech-b100-hang-chinh-hang-p356188.html?spid=49932",
    "https://tiki.vn/chuot-khong-day-sac-dien-hxsj-m103-hang-chinh-hang-p56859344.html?spid=56859348",
    "https://tiki.vn/gang-tay-choi-game-memo-soi-carbon-cao-cap-sieu-ben-co-dan-dan-hoi-tot-do-nhay-cao-hang-nhap-khau-p138489704.html?spid=138489710",
    "https://tiki.vn/mien-phi-12-thang-sim-max-data-4g-vietnamobile-6gb-ngay-tron-goi-1-nam-hang-chinh-hang-p158868703.html?spid=158868704",
    "https://tiki.vn/chuot-khong-day-bluetooth-logitech-pop-mouse-giam-on-nut-emoji-tuy-chinh-hang-chinh-hang-p173058453.html?spid=276344974",
    "https://tiki.vn/tai-nghe-bluetooth-cao-cap-hoco-eq2-5-3-pin-7h-am-thanh-song-dong-bass-cang-hang-chinh-hang-p272030678.html?spid=272030682",
    "https://tiki.vn/sim-4g-mobifone-mdt250a-tron-goi-1-nam-khong-can-nap-tien-hang-chinh-hang-p7734288.html?spid=25411453",
    "https://tiki.vn/chuot-khong-day-logitech-m185-hang-chinh-hang-p54665.html?spid=54672",
    "https://tiki.vn/mieng-lot-chuot-firo-mxl800-extended-tam-lot-chuot-co-lon-2-mat-80cmx40cm-chat-lieu-da-pu-cao-cap-tam-lot-chuot-va-ban-phim-choi-game-tam-trai-ban-lam-viec-hang-chinh-hang-firo-p242665662.html?spid=242665666",
    "https://tiki.vn/may-nghe-nhac-mp3-ruizu-x02-4gb-den-hang-chinh-hang-p275155154.html?spid=2037957"
    "https://tiki.vn/op-lung-chong-soc-lung-trong-phong-cach-moi-danh-cho-iphone-11-12-13-11-pro-12-pro-13-pro-11-pro-max-12-pro-max-13-pro-max-12-mini-13-mini-hang-chinh-hang-p172146754.html?spid=172146774",
    "https://tiki.vn/may-nghe-nhac-mp3-ruizu-x02-4gb-den-hang-chinh-hang-p275155154.html?spid=2037957",
    "https://tiki.vn/tai-nghe-bluetooth-amoi-f9-kem-cu-sac-1a-va-cap-sac-cho-dock-sac-3500mah-hang-chinh-hang-p72928043.html?spid=72928044",
    "https://tiki.vn/may-hut-bui-cam-tay-khong-day-damas-xc628-chinh-hang-p72825272.html?spid=72825416",
    "https://tiki.vn/may-danh-trung-danh-sua-va-tao-bot-cafe-cam-tay-di-dong-3-toc-do-su-dung-pin-sac-cao-cap-hang-chinh-hang-p274541135.html?spid=274541136",
    "https://tiki.vn/may-hut-chan-khong-khong-ken-tui-shineye-p290a-hut-kho-va-hut-uot-cong-suat-manh-me-220w-tang-kem-10-tui-hut-chan-khong-hang-chinh-hang-p70591134.html?spid=94268355",
    "https://tiki.vn/may-hut-chan-khong-tu-dong-khong-ken-tui-p290b-hang-nhap-khau-p65221065.html?spid=93225609",
    "https://tiki.vn/quat-dung-toshiba-f-lsa10-h-vn-50w-xam-hang-chinh-hang-p26078293.html?spid=34240151",
    "https://tiki.vn/may-vat-cam-lock-lock-ejj231-40w-hang-chinh-hang-p273310816.html?spid=10908329",
    "https://tiki.vn/noi-com-dien-tu-toshiba-rc-10nmfvn-wt-1-lit-hang-chinh-hang-p103692716.html?spid=71432",
    "https://tiki.vn/noi-dien-da-nang-lock-lock-ejp116blk-0-8-lit-hang-chinh-hang-p57588481.html?spid=57588482",
    "https://tiki.vn/noi-com-dien-tu-toshiba-rc-18ntfv-w-1-8-lit-hang-chinh-hang-p74890928.html?spid=76195846",
    "https://tiki.vn/binh-dun-sieu-toc-2-lop-lock-lock-ejk738blk-1-7l-hang-chinh-hang-p54716577.html?spid=54716579",
    "https://tiki.vn/binh-dun-sieu-toc-2-lop-lock-lock-ejk738wht-1-7-lit-trang-hang-chinh-hang-p3480333.html?spid=3480335",
    "https://tiki.vn/noi-com-nap-gai-toshiba-rc-10jfm-h-vn-1l-hang-chinh-hang-p2365901.html?spid=2365903"
    "https://tiki.vn/noi-com-nap-gai-toshiba-rc-18jfm-h-vn-1-8l-hang-chinh-hang-p2365981.html?spid=2365983",
    "https://tiki.vn/may-hap-thuc-pham-magic-korea-a64-5-0-lit-hang-chinh-hang-p650250.html?spid=650370",
    "https://tiki.vn/vi-nuong-dien-lock-lock-ejg221-1300w-hang-chinh-hang-p923595.html?spid=924433",
    "https://tiki.vn/bep-hong-ngoai-sunhouse-shd6011-2000w-den-hang-chinh-hang-p530774.html?spid=251421",
    "https://tiki.vn/ban-ui-kho-philips-hd1172-01-1000w-hang-chinh-hang-p392067.html?spid=92969",
    "https://tiki.vn/bep-hong-ngoai-sunhouse-shd6011-2000w-den-hang-chinh-hang-p530774.html?spid=251421",
    "https://tiki.vn/may-hap-thuc-pham-magic-korea-a64-5-0-lit-hang-chinh-hang-p650250.html?spid=650370",
    "https://tiki.vn/may-lam-sua-hat-elmich-cbe-3904-ol-800w-hang-chinh-hang-p271199608.html?spid=271199609",
    "https://tiki.vn/may-vat-cam-philips-hr2738-00-25w-hang-chinh-hang-p394854.html?spid=96204",
    "https://tiki.vn/quat-treo-tuong-toshiba-f-wsa20-h-vn-55w-xam-hang-chinh-hang-p2635459.html?spid=2635461",
    "https://tiki.vn/am-dien-thuy-tinh-sieu-toc-co-dieu-chinh-nhiet-do-lock-lock-ejk341-1-8l-hang-chinh-hang-p4701559.html?spid=4701561",
    "https://tiki.vn/may-hut-bui-lock-lock-env336blk-400w-hang-chinh-hang-p66512832.html?spid=66512833",
    "https://tiki.vn/lo-vi-song-sharp-r-208vn-ws-20l-hang-chinh-hang-p48962414.html?spid=48962415",
    "https://tiki.vn/quat-dung-toshiba-f-lsa10-k-vn-50w-hang-chinh-hang-p34281542.html?spid=34281543",
    "https://tiki.vn/lo-vi-song-sharp-r-205vn-s-20l-hang-chinh-hang-p414764.html?spid=117842",
    "https://tiki.vn/bo-doi-kem-chong-nang-dang-sua-duong-da-kiem-dau-bao-ve-hoan-hao-anessa-gold-milk-60ml-x2-p114271269.html?spid=272278925",
    "https://tiki.vn/nuoc-duong-toc-tinh-dau-buoi-140ml-new-p275970393.html?spid=275970394",
    "https://tiki.vn/nuoc-sen-hau-giang-310ml-p275969483.html?spid=275969484",
    "https://tiki.vn/ca-phe-dak-lak-lam-sach-da-chet-co-the-200ml-p275970279.html?spid=275970280",
    "https://tiki.vn/may-tam-nuoc-khong-day-locknlock-cordless-oral-irrigator-p273598230.html?spid=82721612",
    "https://tiki.vn/xa-phong-tri-mun-lung-va-rua-mat-acnes-washing-bar-75g-p188116913.html?spid=188116914",
    "http://tiki.vn/sua-rua-mat-hada-labo-cho-da-dau-mun-da-nhay-cam-hada-labo-acne-care-calming-cleanser-80g-p174071718.html?spid=174071720",
    "https://tiki.vn/sua-rua-mat-kiem-soat-nhon-ngan-ngua-mun-dang-gel-acnes-oil-control-cleanser-100g-p10640154.html?spid=10862885",
    "https://tiki.vn/son-duong-co-mau-hieu-chinh-sac-moi-lipice-sheer-color-p272129158.html?spid=272129166",
    "https://tiki.vn/kem-chong-nang-dang-sua-diu-nhe-cho-da-nhay-cam-va-tre-em-anessa-perfect-uv-sunscreen-mild-milk-for-sensitive-skin-spf-50-pa-60ml-p77305646.html?spid=272278931",
    "https://tiki.vn/dau-goi-selsun-chong-gau-sach-gau-het-ngua-da-dau-selsun-anti-dandruff-shampoo-250ml-p171564906.html?spid=171564907",
    "https://tiki.vn/sua-rua-mat-duong-trang-hada-labo-perfect-white-tranexamic-acid-cleanser-80g-p54058503.html?spid=54058504",
    "https://tiki.vn/hop-khau-trang-3d-mask-unicharm-nhat-ban-ngan-ngua-khoi-bui-chong-o-nhiem-100-mieng-p35761444.html?spid=35761445",
    "https://tiki.vn/dau-goi-selsun-chong-gau-sach-gau-het-ngua-da-dau-selsun-anti-dandruff-shampoo-50ml-p20541866.html?spid=20541867",
    "https://tiki.vn/sua-rua-mat-ngan-ngua-mun-acnes-creamy-wash-100g-p25108217.html?spid=25108218",
    "https://tiki.vn/sua-rua-mat-kiem-soat-nhon-ngan-ngua-mun-dang-gel-acnes-oil-control-cleanser-100g-p10640154.html?spid=10862885",
    "https://tiki.vn/sua-rua-mat-cho-nam-oxy-sach-sau-giam-mun-kiem-soat-nhon-dang-kem-oxy-total-anti-acne-100g-p22052319.html?spid=22052320",
    "https://tiki.vn/mieng-dan-mun-giup-giam-mun-sung-viem-acnes-clear-patch-24-mieng-p6874429.html?spid=10863062",
    "https://tiki.vn/bao-cao-su-durex-invisible-extra-thin-extra-sensitive-1-hop-10-bao-p1672157.html?spid=1115608",
    "https://tiki.vn/bao-cao-su-durex-kingtex-3-bao-p1672155.html?spid=85357",
    "https://tiki.vn/giay-tham-dau-kiem-soat-nhon-ngua-mun-acnes-oil-remover-paper-100-to-p1634981.html?spid=10862722",
    "https://tiki.vn/dau-goi-selsun-chong-gau-sach-gau-het-ngua-da-dau-selsun-anti-dandruff-shampoo-100ml-p632680.html?spid=10862721",
    "https://tiki.vn/sua-rua-mat-duong-am-toi-uu-hada-labo-advanced-nourish-cleanser-80g-p631034.html?spid=10862753",
    "https://tiki.vn/bao-cao-su-durex-fetherlite-hop-12-bao-p385582.html?spid=85518",
    "https://tiki.vn/kem-chong-nang-skin-aqua-duong-trang-kiem-dau-dung-hang-ngay-dang-sua-sunplay-skin-aqua-clear-white-eco-viet-nam-spf50-pa-55g-p630778.html?spid=10862294",
    "https://tiki.vn/bao-cao-su-durex-kingtex-12-bao-p767101.html?spid=1115606",
    "https://tiki.vn/sua-duong-the-nivea-phuc-hoi-chong-nang-ban-ngay-200-ml-88310-p275342473.html?spid=275342474",
    "https://tiki.vn/dung-dich-chong-muoi-dang-phun-suong-remos-mentholatum-huong-sa-chanh-70ml-p276862669.html?spid=276862670",
    "https://tiki.vn/sua-rua-mat-cho-nam-oxy-sach-sau-giam-nhon-dang-kem-co-hat-massage-oxy-deep-wash-100g-p22052322.html?spid=22052324",
    "https://tiki.vn/nuoc-nho-mat-cap-am-bo-sung-nuoc-mat-nhan-tao-v-rohto-dryeye-13ml-p270688219.html?spid=270688220",
    "https://tiki.vn/kem-lam-mo-seo-tham-seo-ro-seo-lom-scar-esthetique-cua-rejuvaskin-thuong-hieu-ho-tro-tri-seo-hoa-ky-p260844985.html?spid=38705976",
    "https://tiki.vn/sua-rua-mat-cho-nam-oxy-sach-sau-mat-lanh-dang-kem-oxy-perfect-wash-100g-p22052325.html?spid=22052326",
    "https://tiki.vn/kem-mo-seo-va-tham-dang-gel-acnes-scar-care-12g-p631959.html?spid=10862535",
    "https://tiki.vn/bao-cao-su-durex-fetherlite-ultima-hop-12-bao-p385477.html?spid=85399",
    "https://tiki.vn/kem-lam-mo-seo-tham-seo-ro-seo-lom-scar-esthetique-cua-rejuvaskin-thuong-hieu-ho-tro-tri-seo-hoa-ky-p260844985.html?spid=38705976",
    "https://tiki.vn/lan-ngan-mui-nivea-men-dry-impact-kho-thoang-50ml-81610-p273455553.html?spid=273455554",
    "https://tiki.vn/xit-ngan-mui-nivea-men-dry-impact-kho-thoang-150ml-81602-p54384691.html?spid=273568735",
    "https://tiki.vn/sua-rua-mat-tri-mun-senka-perfect-whip-acne-care-100g-p33785195.html?spid=44894558",
    "https://tiki.vn/kem-duong-cai-thien-lao-hoa-hada-labo-pro-aging-retinol-b3-cream-50g-p631498.html?spid=10862543",
    "https://tiki.vn/sua-rua-mat-min-va-san-chac-da-senka-perfect-whip-collagen-in-120g-4901872462087-p2123245.html?spid=2123247",
    "https://tiki.vn/sua-rua-mat-duong-trang-dang-kem-acnes-whitening-cleanser-100g-p10638708.html?spid=10862754",
    "https://tiki.vn/durex-gel-boi-tron-play-massage-2-in-1-200ml-p385574.html?spid=85509",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-vi-cam-170ml-hop-p2454355.html?spid=2600531",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-vi-dau-170ml-hop-p2454353.html?spid=2600529",
    "https://tiki.vn/tang-ly-su-cao-cap-ca-phe-hoa-tan-nescafe-cafe-viet-35-goi-vi-manh-dac-trung-p277838067.html?spid=277838068",
    "https://tiki.vn/tang-ly-su-cao-cap-ca-phe-hoa-tan-nescafe-cafe-viet-35-goi-vi-manh-dac-trung-p277838067.html?spid=277838068",
    "https://tiki.vn/tang-hop-2-ly-thuy-tinh-combo-2-bich-ca-phe-den-hoa-tan-nescafe-cafe-viet-tui-35-goi-x-16g-p277779078.html?spid=277779079",
    "https://tiki.vn/tang-ly-su-cao-cap-ca-phe-hoa-tan-nescafe-cafe-viet-35-goi-vi-manh-dac-trung-p277838067.html?spid=277838068",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-bac-ha-viet-quat-170ml-hop-p2454357.html?spid=2600533",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-trai-luu-170ml-hop-p2454359.html?spid=2600535",
    "https://tiki.vn/tang-bo-3-ly-thuy-tinh-cao-cap-nescafe-combo-2-bich-nescafe-3in1-cong-thuc-cai-tien-vi-nguyen-ban-bich-46-goi-p274798897.html?spid=274798898",
    "https://tiki.vn/sua-lua-mach-nestle-milo-thung-48-hop-roi-x-180ml-khong-mang-nhua-48x180ml-p274273463.html?spid=274273464",
    "https://tiki.vn/tang-bo-binh-ly-lumiarc-750ml-200ml-nescafe-combo-3-bich-ca-phe-den-hoa-tan-nescafe-cafe-viet-tui-35-goi-x-16g-p274086107.html?spid=274086111",
    "https://tiki.vn/ca-phe-hoa-tan-nescafe-3in1-vi-nguyen-ban-cong-thuc-cai-tien-bich-46-goi-x-16g-p272635007.html?spid=272635008",
    "https://tiki.vn/thung-24-hop-sua-9-loai-hat-vinamilk-super-nut-180ml-p176103097.html?spid=176103098",
    "https://tiki.vn/thung-48-hop-sua-nestle-milo-it-duong-180ml-hop-p15973974.html?spid=15973975",
    "https://tiki.vn/trung-nguyen-legend-ca-phe-hoa-tan-rang-xay-3in1-classic-bich-50-goi-x-17gr-p161454715.html?spid=161454716",
    "https://tiki.vn/dau-dau-nanh-simply-1l-2l-5l-p133696713.html?spid=196326",
    "https://tiki.vn/banh-gao-lut-nguyen-hat-gufoods-510g-54-banh-phu-hop-an-kieng-tap-gym-eat-clean-p58811363.html?spid=209906030",
    "https://tiki.vn/trung-nguyen-legend-ca-phe-hoa-tan-rang-xay-3in1-cafe-sua-da-hop-9-goi-x-25gr-p58451648.html?spid=58451649",
    "https://tiki.vn/trung-nguyen-legend-ca-phe-hoa-tan-g7-3in1-bich-100-sticks-x-16gr-goi-dai-p58434660.html?spid=58434661",
    "https://tiki.vn/trung-nguyen-legend-ca-phe-hoa-tan-rang-xay-3in1-special-edition-hop-18-goi-x-25gr-p51713363.html?spid=59487184",
    "https://tiki.vn/thung-48-hop-sua-nestle-milo-it-duong-180ml-hop-p15973974.html?spid=15973975",
    "https://tiki.vn/thung-sua-dau-nanh-fami-nguyen-chat-200ml-x-36-hop-p12629696.html?spid=12629697",
    "https://tiki.vn/thung-48-hop-sua-uong-dutch-lady-co-gai-ha-lan-co-duong-cao-khoe-48x170ml-p13409520.html?spid=13409521",
    "https://tiki.vn/thung-48-hop-sua-tuoi-tiet-trung-dutch-lady-co-gai-ha-lan-co-duong-48x180ml-p2454283.html?spid=2599793",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-vi-cam-170ml-hop-p2454355.html?spid=2600531",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-vi-dau-170ml-hop-p2454353.html?spid=2600529",
    "https://tiki.vn/thung-48-hop-sua-tuoi-tiet-trung-dutch-lady-co-gai-ha-lan-co-duong-48x180ml-p2454283.html?spid=2599793",
    "https://tiki.vn/nuoc-yen-sao-khanh-hoa-sanest-dong-lo-70ml-p138956444.html?spid=138956448",
    "https://tiki.vn/thung-48-hop-sua-chua-len-men-tu-nhien-yomost-vi-cam-170ml-hop-p2454355.html?spid=2600531",
    "https://tiki.vn/dau-olive-pomace-silarus-1l-p5032435.html?spid=5032437",
    "https://tiki.vn/giay-tham-dau-kiem-soat-nhon-ngua-mun-acnes-oil-remover-paper-100-to-p1634981.html?spid=10862722",
    "https://tiki.vn/dau-goi-selsun-chong-gau-sach-gau-het-ngua-da-dau-selsun-anti-dandruff-shampoo-100ml-p632680.html?spid=10862721",
    "https://tiki.vn/sua-rua-mat-duong-am-toi-uu-hada-labo-advanced-nourish-cleanser-80g-p631034.html?spid=10862753",
    "https://tiki.vn/bao-cao-su-durex-fetherlite-hop-12-bao-p385582.html?spid=85518",
    "https://tiki.vn/kem-chong-nang-skin-aqua-duong-trang-kiem-dau-dung-hang-ngay-dang-sua-sunplay-skin-aqua-clear-white-eco-viet-nam-spf50-pa-55g-p630778.html?spid=10862294",
    "https://tiki.vn/bao-cao-su-durex-kingtex-12-bao-p767101.html?spid=1115606",
    "https://tiki.vn/sua-duong-the-nivea-phuc-hoi-chong-nang-ban-ngay-200-ml-88310-p275342473.html?spid=275342474",
    "https://tiki.vn/dung-dich-chong-muoi-dang-phun-suong-remos-mentholatum-huong-sa-chanh-70ml-p276862669.html?spid=276862670",
    "https://tiki.vn/sua-rua-mat-cho-nam-oxy-sach-sau-giam-nhon-dang-kem-co-hat-massage-oxy-deep-wash-100g-p22052322.html?spid=22052324",
    "https://tiki.vn/nuoc-nho-mat-cap-am-bo-sung-nuoc-mat-nhan-tao-v-rohto-dryeye-13ml-p270688219.html?spid=270688220",
    "https://tiki.vn/kem-lam-mo-seo-tham-seo-ro-seo-lom-scar-esthetique-cua-rejuvaskin-thuong-hieu-ho-tro-tri-seo-hoa-ky-p260844985.html?spid=38705976",
    "https://tiki.vn/sua-rua-mat-cho-nam-oxy-sach-sau-mat-lanh-dang-kem-oxy-perfect-wash-100g-p22052325.html?spid=22052326",
    "https://tiki.vn/kem-mo-seo-va-tham-dang-gel-acnes-scar-care-12g-p631959.html?spid=10862535",
    "https://tiki.vn/bao-cao-su-durex-fetherlite-ultima-hop-12-bao-p385477.html?spid=85399"
]

for u in urls:
    print("="*80)
    auto_crawl_from_url(u)
    time.sleep(3)


🔍 product_id = 197214029, spid = 197214037
✅ Dùng product_id = 197214029 để crawl reviews...
Đã crawl trang 1 (20 bình luận)
Đã crawl trang 2 (20 bình luận)
Đã crawl trang 3 (20 bình luận)
Đã crawl trang 4 (20 bình luận)
Đã crawl trang 5 (20 bình luận)
Đã crawl trang 6 (20 bình luận)
Đã crawl trang 7 (20 bình luận)
Đã crawl trang 8 (20 bình luận)
Đã crawl trang 9 (9 bình luận)
📁 Đã lưu tiki_reviews.csv với 169 bình luận.
🔍 product_id = 87960372, spid = 87960373
✅ Dùng product_id = 87960372 để crawl reviews...
Đã crawl trang 1 (20 bình luận)
Đã crawl trang 2 (20 bình luận)
Đã crawl trang 3 (20 bình luận)
Đã crawl trang 4 (20 bình luận)
Đã crawl trang 5 (20 bình luận)
Đã crawl trang 6 (20 bình luận)
Đã crawl trang 7 (20 bình luận)
Đã crawl trang 8 (20 bình luận)
Đã crawl trang 9 (20 bình luận)
Đã crawl trang 10 (20 bình luận)
📁 Đã lưu tiki_reviews.csv với 200 bình luận.
🔍 product_id = 101782486, spid = 101782488
✅ Dùng product_id = 101782486 để crawl reviews...
Đã crawl trang 1 (20 bình 